# Customer Churn Exploratory Data Analysis (EDA)
**Project:** Churn MLOps Platform  
**Dataset:** IBM Telco Customer Churn  

### Objectives:
1. Understand dataset structure, missing data patterns, and target class imbalance.
2. Analyze customer churn rates across contract types, services, tenure, and monthly charges.
3. Validate hypotheses for feature engineering (tenure buckets, charge ratios, service counts).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

DATA_PATH = '../data/telco_churn.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(5)


## 1. Data Cleaning & Type Corrections
Inspect missing values and convert `TotalCharges` from whitespace string to numeric float.


In [ ]:
# Check for blank strings in TotalCharges
blank_mask = df['TotalCharges'].astype(str).str.strip() == ''
print(f'Blank TotalCharges rows count: {blank_mask.sum()}')

# Convert to float and impute tenure=0 cases with 0.0
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].astype(str).str.strip(), errors='coerce').fillna(0.0)
df['Churn_Bool'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.info()


## 2. Target Variable Distribution: Class Imbalance
Examine overall churn rate.


In [ ]:
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values, palette=['#3b82f6', '#ef4444'], ax=ax)
ax.set_title(f"Churn Distribution (Total Churned: {churn_pct['Yes']:.1f}%)")
ax.set_ylabel('Number of Customers')
plt.show()


## 3. Tenure & Monthly Charges Analysis
How does tenure and pricing correlate with churn risk?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution by Churn
sns.histplot(data=df, x='tenure', hue='Churn', kde=True, bins=30, ax=axes[0], palette=['#3b82f6', '#ef4444'])
axes[0].set_title('Customer Tenure Distribution by Churn Status')

# Monthly Charges by Churn
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', palette=['#3b82f6', '#ef4444'], ax=axes[1])
axes[1].set_title('Monthly Charges Distribution by Churn Status')
plt.tight_layout()
plt.show()


## 4. Contract Type & Payment Method Impact
Contract duration and automated payment methods are historically the strongest drivers of churn.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contract Churn Rate
contract_churn = df.groupby('Contract')['Churn_Bool'].mean().reset_index()
sns.barplot(data=contract_churn, x='Contract', y='Churn_Bool', palette='Blues_r', ax=axes[0])
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Proportion')

# Payment Method Churn Rate
payment_churn = df.groupby('PaymentMethod')['Churn_Bool'].mean().reset_index()
sns.barplot(data=payment_churn, x='PaymentMethod', y='Churn_Bool', palette='Reds_r', ax=axes[1])
axes[1].set_title('Churn Rate by Payment Method')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].set_ylabel('Churn Proportion')
plt.tight_layout()
plt.show()


## 5. Feature Engineering Validation
Verify interaction features:
- `charges_per_tenure`
- `num_services` count


In [ ]:
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['num_services'] = df[services].apply(lambda row: sum(row == 'Yes'), axis=1)
df['charges_per_tenure'] = df['TotalCharges'] / (df['tenure'] + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='Churn', y='num_services', palette=['#3b82f6', '#ef4444'], ax=axes[0])
axes[0].set_title('Number of Subscribed Services vs Churn')

sns.boxplot(data=df, x='Churn', y='charges_per_tenure', palette=['#3b82f6', '#ef4444'], ax=axes[1])
axes[1].set_title('Charges per Tenure Month vs Churn')
plt.tight_layout()
plt.show()


## Key Findings & Modeling Recommendations
1. **Class Imbalance:** Churn rate is ~26.5%. Stratified cross-validation and SMOTE / class weighting are required.
2. **Early Churn Risk:** Churn is heavily concentrated in the first 0-12 months of tenure.
3. **Month-to-month Contracts:** Customers on month-to-month contracts churn at ~42%, compared to <3% for 2-year contracts.
4. **Electronic Check:** Customers paying via Electronic Check show significantly higher churn.
5. **Feature Engineering:** `charges_per_tenure`, `num_services`, and tenure buckets provide strong discriminative power.
